# Benchmark: drivers of the mean and of the spread

A distributional regression models more than the mean. When the spread of
the response changes with the predictors, the analyst has two questions:
which columns move the mean, which move the spread, and what shape does each
effect take? This study measures how well each method answers both when it
is given every column for both parameters and no hint of which matter.

Twelve predictors are independent and uniform on $(-1, 1)$. The response is
Gaussian with

$$
\mu(x) = \sin \pi x_1 + 0.8 x_2, \qquad
\log \sigma(x) = -0.3 + 0.6 x_3 - 0.8\,(x_4^2 - 1/3),
$$

so $x_1$ and $x_2$ drive the mean, $x_3$ and $x_4$ the standard deviation,
and $x_5$ to $x_{12}$ neither. Training samples have 1,000 or 3,000 rows.

The methods:

* **mboostrs**: `gamboostLSS` with a P-spline for every column in both
  parameters, fitted with `method="noncyclic"` (each step updates the one
  term of one parameter that most reduces the negative log-likelihood) and
  the total step count cross-validated; the same followed by `deselect`,
  which drops every term under 1% of the risk reduction over both
  parameters; and that followed by `gamlss_refit`, which refits the kept
  terms jointly with REML smoothing parameters.
* **mgcv**: `gam(list(y ~ s(x1) + ..., ~ s(x1) + ...), family = gaulss(),
  select = TRUE)`, a term kept at p < 0.05.
* **gamlss**: a P-spline `pb()` for every column in both parameters, no
  selection.
* **NGBoost**: natural-gradient boosting of trees for both parameters.

:::{note} Recorded before the response-units change
The mboostrs `gamboostLSS` results on this page are recorded results. They were recorded with mboostrs 0.2.3 or earlier, when distributional (LSS) and `Quantile` fits boosted the response in the units it was given. Since then these fits divide the response by its standard deviation, boost the rescaled response and map the fitted model back (`response_scale="auto"`), so a step has the same size in any units; `Quantile` also starts at the `tau`-quantile of the response, and takes steps of 0.3 when no control is given. Refitting on the current version gives different numbers. `response_scale=None` with the settings the page states reproduces the recorded fits; [Response units](../implementation/parity_and_bugs.ipynb#response-units) describes the change and its measured effect.
:::

:::{admonition} What this study estimates
:class: dropdown

| | |
|---|---|
| **Target** | the columns that drive each parameter, and the functions $\mu(x)$ and $\log\sigma(x)$ |
| **Observed response** | simulated Gaussian with predictor-dependent mean and standard deviation |
| **Estimand** | per parameter, the drivers found and the false ones kept; RMSE of $\mu(x)$ and of $\log\sigma(x)$ on 3,000 test rows; test negative log-likelihood per row |
| **Units** | response units ($\mu$), log units ($\sigma$), nats per row |
| **Sampling unit** | one row; 1,000 or 3,000 training rows per seed |
| **Dependence** | independent rows and predictors |
| **Split / resampling unit** | a fresh test sample from the same design |
| **Tuning** | mboostrs: 5-fold CV of the total step count (grid to 3,000); mgcv, gamlss: REML / local ML; NGBoost: defaults |
| **Uncertainty** | mean and standard error over 10 seeds |
| **Claim type** | methodological |
:::

The results come from `benches/bench_lss_drivers.py`
(`python benches/bench_lss_drivers.py --out docs/book/cases/bench_lss_drivers.json`).
One record shows the fields.

In [1]:
import json
from pathlib import Path

import polars as pl

bench = json.loads(Path("bench_lss_drivers.json").read_text())
records = [r for r in bench["results"] if "error" not in r and "failure" not in r]
failed = [r for r in bench["results"] if "error" in r or "failure" in r]
example = next(r for r in records if r["method"] == "mboostrs LSS + deselect + REML refit")
{k: (round(v, 3) if isinstance(v, float) else v) for k, v in example.items()}

{'task': 'n1000',
 'method': 'mboostrs LSS + deselect + REML refit',
 'seed': 0,
 'seconds': 12.621,
 'mu_rmse': 0.063,
 'log_sigma_rmse': 0.047,
 'nll': 1.125,
 'mu_selected': ['x1', 'x2'],
 'mu_found': 2,
 'mu_false': 0,
 'mu_exact': True,
 'sigma_selected': ['x3', 'x4'],
 'sigma_found': 2,
 'sigma_false': 0,
 'sigma_exact': True,
 '_status': 'success',
 '_identity': {'fingerprint': '6e25e69945fa1aa3e06087c6d3bae84cd6971a1f53d4cb7e537021399141cd41',
  'components': {'runner_modules': ['_jobs',
    'bench_lss_drivers',
    'benchmark_contract'],
   'runner_hash': 'cb517bd56a978653b2910192615486d9758b91271828a8b94d02f0909ac3b730',
   'environment': {'python': '3.12.11',
    'executable': '/Users/fcela/src/public_github/mboostrs-dev/.venv/bin/python',
    'platform': 'macOS-26.3-arm64-arm-64bit',
    'machine': 'arm64',
    'packages': {'numpy': '2.4.6',
     'polars': '1.41.0',
     'pandas': '2.3.3',
     'scipy': '1.16.3',
     'scikit-learn': '1.9.0',
     'lightgbm': '4.7.0',
     

In [2]:
SCORES = ["mu_found", "mu_false", "mu_exact", "sigma_found", "sigma_false",
          "sigma_exact", "mu_rmse", "log_sigma_rmse", "nll", "seconds"]
flat = pl.DataFrame([{k: r.get(k) for k in ["task", "method", "seed"] + SCORES}
                     for r in records], infer_schema_length=None)
flat = flat.with_columns(pl.col("mu_exact", "sigma_exact").cast(pl.Float64))
table = flat.group_by("task", "method").agg(
    [pl.col(c).cast(pl.Float64).mean().alias(c) for c in SCORES])


def value(task, method, column):
    row = table.filter((pl.col("task") == task) & (pl.col("method") == method))
    return float(row[column][0])


from bench_tables import table as annotated

table = table.with_columns(
    pl.col("task").str.replace("n", "").cast(pl.Int64)
    .map_elements(lambda n: f"n = {n:,} training rows", return_dtype=pl.String).alias("size"))

In [3]:
display(annotated(
    table.sort("size", "log_sigma_rmse").select(
        "size", "method", "mu_found", "mu_false", "mu_exact",
        "sigma_found", "sigma_false", "sigma_exact"),
    title="Drivers of the mean and of the spread found by each method",
    subtitle="Mean over 10 seeds; 12 predictors, 2 drive μ and 2 drive σ",
    group="size",
    spanners={"Mean μ": ["mu_found", "mu_false", "mu_exact"],
              "Standard deviation σ": ["sigma_found", "sigma_false", "sigma_exact"]},
    labels={"mu_found": "found ↑", "mu_false": "false ↓", "mu_exact": "exact ↑",
            "sigma_found": "found ↑", "sigma_false": "false ↓", "sigma_exact": "exact ↑"},
    better={"mu_false": "min", "mu_exact": "max", "sigma_false": "min", "sigma_exact": "max"},
    formats={"mu_found": 1, "sigma_found": 1, "mu_false": 1, "sigma_false": 1,
             "mu_exact": 1, "sigma_exact": 1},
    notes=["found: true drivers of the parameter (of 2: x1, x2 for μ; x3, x4 for σ) among "
           "the columns the method keeps for it. false: kept columns that do not drive it "
           "(of 10). exact: share of seeds in which the kept set is exactly the true one. "
           "A method keeps a column by its own rule: a selected term for mboostrs, "
           "p < 0.05 for mgcv. – : the method has no selection rule and uses every "
           "column (gamlss, NGBoost). Bold: best within "
           "each training size."]))

GT(_tbl_data=shape: (12, 8)
┌─────────────┬────────────┬──────────┬──────────┬──────────┬────────────┬────────────┬────────────┐
│ size        ┆ method     ┆ mu_found ┆ mu_false ┆ mu_exact ┆ sigma_foun ┆ sigma_fals ┆ sigma_exac │
│ ---         ┆ ---        ┆ ---      ┆ ---      ┆ ---      ┆ d          ┆ e          ┆ t          │
│ str         ┆ str        ┆ f64      ┆ f64      ┆ f64      ┆ ---        ┆ ---        ┆ ---        │
│             ┆            ┆          ┆          ┆          ┆ f64        ┆ f64        ┆ f64        │
╞═════════════╪════════════╪══════════╪══════════╪══════════╪════════════╪════════════╪════════════╡
│ n = 1,000   ┆ mboostrs   ┆ 2.0      ┆ 0.0      ┆ 1.0      ┆ 2.0        ┆ 0.2        ┆ 0.8        │
│ training    ┆ LSS +      ┆          ┆          ┆          ┆            ┆            ┆            │
│ rows        ┆ deselect + ┆          ┆          ┆          ┆            ┆            ┆            │
│             ┆ REML…      ┆          ┆          ┆          ┆            ┆            ┆            │
│ n = 1,000   ┆ mboostrs   ┆ 2.0      ┆ 0.0      ┆ 1.0      ┆ 2.0        ┆ 0.2        ┆ 0.8        │
│ training    ┆ LSS +      ┆          ┆          ┆          ┆            ┆            ┆            │
│ rows        ┆ deselect   ┆          ┆          ┆          ┆            ┆            ┆            │
│ n = 1,000   ┆ mgcv       ┆ 2.0      ┆ 1.0      ┆ 0.4      ┆ 2.0        ┆ 1.4        ┆ 0.2        │
│ training    ┆ gaulss     ┆          ┆          ┆          ┆            ┆            ┆            │
│ rows        ┆ select     ┆          ┆          ┆          ┆            ┆            ┆            │
│ n = 1,000   ┆ mboostrs   ┆ 2.0      ┆ 4.0      ┆ 0.4      ┆ 2.0        ┆ 4.2        ┆ 0.1        │
│ training    ┆ LSS (CV)   ┆          ┆          ┆          ┆            ┆            ┆            │
│ rows        ┆            ┆          ┆          ┆          ┆            ┆            ┆            │
│ n = 1,000   ┆ gamlss pb  ┆ null     ┆ null     ┆ null     ┆ null       ┆ null       ┆ null       │
│ training    ┆            ┆          ┆          ┆          ┆            ┆            ┆            │
│ rows        ┆            ┆          ┆          ┆          ┆            ┆            ┆            │
│ …           ┆ …          ┆ …        ┆ …        ┆ …        ┆ …          ┆ …          ┆ …          │
│ n = 3,000   ┆ mboostrs   ┆ 2.0      ┆ 0.0      ┆ 1.0      ┆ 2.0        ┆ 0.0        ┆ 1.0        │
│ training    ┆ LSS +      ┆          ┆          ┆          ┆            ┆            ┆            │
│ rows        ┆ deselect   ┆          ┆          ┆          ┆            ┆            ┆            │
│ n = 3,000   ┆ mgcv       ┆ 2.0      ┆ 0.5      ┆ 0.6      ┆ 2.0        ┆ 0.3        ┆ 0.7        │
│ training    ┆ gaulss     ┆          ┆          ┆          ┆            ┆            ┆            │
│ rows        ┆ select     ┆          ┆          ┆          ┆            ┆            ┆            │
│ n = 3,000   ┆ mboostrs   ┆ 2.0      ┆ 9.2      ┆ 0.0      ┆ 2.0        ┆ 7.9        ┆ 0.0        │
│ training    ┆ LSS (CV)   ┆          ┆          ┆          ┆            ┆            ┆            │
│ rows        ┆            ┆          ┆          ┆          ┆            ┆            ┆            │
│ n = 3,000   ┆ gamlss pb  ┆ null     ┆ null     ┆ null     ┆ null       ┆ null       ┆ null       │
│ training    ┆            ┆          ┆          ┆          ┆            ┆            ┆            │
│ rows        ┆            ┆          ┆          ┆          ┆            ┆            ┆            │
│ n = 3,000   ┆ NGBoost    ┆ null     ┆ null     ┆ null     ┆ null       ┆ null       ┆ null       │
│ training    ┆            ┆          ┆          ┆          ┆            ┆            ┆            │
│ rows        ┆            ┆          ┆          ┆          ┆            ┆            ┆            │
└─────────────┴────────────┴──────────┴──────────┴──────────┴────────────┴────────────┴────────────┘, _body=<great_tables._gt_data.Bod

In [4]:
display(annotated(
    table.sort("size", "log_sigma_rmse").select(
        "size", "method", "mu_rmse", "log_sigma_rmse", "nll", "seconds"),
    title="Accuracy of the fitted mean, spread and predictive density",
    subtitle="Mean over 10 seeds; 3,000 test rows",
    group="size",
    labels={"mu_rmse": "μ RMSE ↓", "log_sigma_rmse": "log σ RMSE ↓", "nll": "NLL ↓",
            "seconds": "s"},
    better={"mu_rmse": "min", "log_sigma_rmse": "min", "nll": "min"},
    formats={"seconds": 1},
    notes=["μ RMSE: root mean squared error of the fitted mean against the true μ(x), in "
           "response units. log σ RMSE: the same for log σ(x). NLL: mean negative "
           "log-likelihood of the test responses, in nats per row. s: mean seconds per fit "
           "including tuning. Bold: best within each training size. Source: "
           "bench_lss_drivers.json, written by benches/bench_lss_drivers.py."]))
f"{len(records)} fits, {len(failed)} failed"

GT(_tbl_data=shape: (12, 6)
┌───────────────────────┬──────────────────────┬──────────┬────────────────┬──────────┬────────────┐
│ size                  ┆ method               ┆ mu_rmse  ┆ log_sigma_rmse ┆ nll      ┆ seconds    │
│ ---                   ┆ ---                  ┆ ---      ┆ ---            ┆ ---      ┆ ---        │
│ str                   ┆ str                  ┆ f64      ┆ f64            ┆ f64      ┆ f64        │
╞═══════════════════════╪══════════════════════╪══════════╪════════════════╪══════════╪════════════╡
│ n = 1,000 training    ┆ mboostrs LSS +       ┆ 0.058367 ┆ 0.059974       ┆ 1.130351 ┆ 11.18427   │
│ rows                  ┆ deselect + REML…     ┆          ┆                ┆          ┆            │
│ n = 1,000 training    ┆ mboostrs LSS +       ┆ 0.058275 ┆ 0.072193       ┆ 1.131252 ┆ 11.146407  │
│ rows                  ┆ deselect             ┆          ┆                ┆          ┆            │
│ n = 1,000 training    ┆ mgcv gaulss select   ┆ 0.080767 ┆ 0.083504       ┆ 1.13755  ┆ 310.167746 │
│ rows                  ┆                      ┆          ┆                ┆          ┆            │
│ n = 1,000 training    ┆ mboostrs LSS (CV)    ┆ 0.104697 ┆ 0.097961       ┆ 1.142829 ┆ 10.864032  │
│ rows                  ┆                      ┆          ┆                ┆          ┆            │
│ n = 1,000 training    ┆ gamlss pb            ┆ 0.103811 ┆ 0.119558       ┆ 1.15207  ┆ 19.776822  │
│ rows                  ┆                      ┆          ┆                ┆          ┆            │
│ …                     ┆ …                    ┆ …        ┆ …              ┆ …        ┆ …          │
│ n = 3,000 training    ┆ mboostrs LSS +       ┆ 0.037113 ┆ 0.034263       ┆ 1.12453  ┆ 18.785993  │
│ rows                  ┆ deselect             ┆          ┆                ┆          ┆            │
│ n = 3,000 training    ┆ mgcv gaulss select   ┆ 0.044127 ┆ 0.039242       ┆ 1.126082 ┆ 801.588275 │
│ rows                  ┆                      ┆          ┆                ┆          ┆            │
│ n = 3,000 training    ┆ mboostrs LSS (CV)    ┆ 0.057682 ┆ 0.05335        ┆ 1.128699 ┆ 17.303807  │
│ rows                  ┆                      ┆          ┆                ┆          ┆            │
│ n = 3,000 training    ┆ gamlss pb            ┆ 0.057448 ┆ 0.05557        ┆ 1.129531 ┆ 22.189696  │
│ rows                  ┆                      ┆          ┆                ┆          ┆            │
│ n = 3,000 training    ┆ NGBoost              ┆ 0.169515 ┆ 0.145121       ┆ 1.181409 ┆ 17.966148  │
│ rows                  ┆                      ┆          ┆                ┆          ┆            │
└───────────────────────┴──────────────────────┴──────────┴────────────────┴──────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x109d5f860>, _boxhead=Boxhead([ColInfo(var='size', type=<ColInfoTypeEnum.row_group: 3>, column_label='size', column_align='left', column_width=None), ColInfo(var='method', type=<ColInfoTypeEnum.stub: 2>, column_label='method', column_align='left', column_width=None), ColInfo(var='mu_rmse', type=<ColInfoTypeEnum.default: 1>, column_label='μ RMSE ↓', column_align='right', column_width=None), ColInfo(var='log_sigma_rmse', type=<ColInfoTypeEnum.default: 1>, column_label='log σ RMSE ↓', column_align='right', column_width=None), ColInfo(var='nll', type=<ColInfoTypeEnum.default: 1>, column_label='NLL ↓', column_align='right', column_width=None), ColInfo(var='seconds', type=<ColInfoTypeEnum.default: 1>, column_label='s', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x109d5fb60>, _spanners=Spanners([]), _heading=Heading(title='Accuracy of the fitted mean, spread and predictive density', subtitle='Mean over 10 seeds; 3,000 test rows', preheader=None), _stubhead='Method', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10e1a9430>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10e1a94c0>, _source_notes=['μ RMSE: root

'120 fits, 0 failed'

(drivers-found)=
## Selected drivers

Every method that selects finds both drivers of each parameter in every
seed; they differ in the columns they add. The cross-validated boosting fit
keeps many: at 3,000 rows it updates
{eval}`round(value("n3000", "mboostrs LSS (CV)", "mu_false"), 1)` noise columns
in the mean and {eval}`round(value("n3000", "mboostrs LSS (CV)", "sigma_false"), 1)`
in the spread, because every term a step touches once stays in the path.
Deselection removes them: after it, the selected set is exactly
$\{x_1, x_2\}$ for the mean in
{eval}`f"{value('n3000', 'mboostrs LSS + deselect', 'mu_exact'):.0%}"` of the
seeds at 3,000 rows and exactly $\{x_3, x_4\}$ for the spread in
{eval}`f"{value('n3000', 'mboostrs LSS + deselect', 'sigma_exact'):.0%}"`; at
1,000 rows the spread's set is exact in
{eval}`f"{value('n1000', 'mboostrs LSS + deselect', 'sigma_exact'):.0%}"`.
mgcv's double penalty (a penalty on each smooth's linear part, so that a whole term can shrink to zero) with a 5% p-value cut keeps
{eval}`round(value("n1000", "mgcv gaulss select", "mu_false"), 1)` false
columns in the mean and {eval}`round(value("n1000", "mgcv gaulss select", "sigma_false"), 1)`
in the spread at 1,000 rows, and
{eval}`round(value("n3000", "mgcv gaulss select", "mu_false"), 1)` and
{eval}`round(value("n3000", "mgcv gaulss select", "sigma_false"), 1)` at
3,000. A p-value per term is a test of each smooth separately, so about one in
twenty noise terms passes at the 5% level.

## Accuracy of the two functions

The REML refit of the deselected terms estimates both functions most
accurately: at 3,000 rows its error in $\mu(x)$ is
{eval}`round(value("n3000", "mboostrs LSS + deselect + REML refit", "mu_rmse"), 3)`
and in $\log\sigma(x)$
{eval}`round(value("n3000", "mboostrs LSS + deselect + REML refit", "log_sigma_rmse"), 3)`,
against {eval}`round(value("n3000", "mgcv gaulss select", "mu_rmse"), 3)` and
{eval}`round(value("n3000", "mgcv gaulss select", "log_sigma_rmse"), 3)` for
mgcv, {eval}`round(value("n3000", "gamlss pb", "mu_rmse"), 3)` and
{eval}`round(value("n3000", "gamlss pb", "log_sigma_rmse"), 3)` for gamlss and
{eval}`round(value("n3000", "NGBoost", "mu_rmse"), 3)` and
{eval}`round(value("n3000", "NGBoost", "log_sigma_rmse"), 3)` for NGBoost. The
deselected boosting fit alone is close to the refit; the refit removes the
shrinkage that early stopping leaves on each effect. Removing the noise
columns is also what makes the functions accurate: the cross-validated fit,
which keeps them, has errors
{eval}`round(value("n3000", "mboostrs LSS (CV)", "mu_rmse"), 3)` and
{eval}`round(value("n3000", "mboostrs LSS (CV)", "log_sigma_rmse"), 3)`. The
test negative log-likelihood orders the methods the same way, with smaller
differences, because it is dominated by the noise every method leaves.

mgcv takes {eval}`round(value("n3000", "mgcv gaulss select", "seconds"))` seconds
per fit at 3,000 rows with 24 smooths, the refit route
{eval}`round(value("n3000", "mboostrs LSS + deselect + REML refit", "seconds"))`
(boosting path, cross-validation, deselection and REML refit together); the times describe the order of cost.

## Summary

| Question | mboostrs LSS + deselect + REML refit | Next best |
|---|---|---|
| Drivers of the mean, exact set (3,000 rows) | {eval}`f"{value('n3000', 'mboostrs LSS + deselect + REML refit', 'mu_exact'):.0%}"` of seeds | mgcv {eval}`f"{value('n3000', 'mgcv gaulss select', 'mu_exact'):.0%}"` |
| Drivers of the spread, exact set (3,000 rows) | {eval}`f"{value('n3000', 'mboostrs LSS + deselect + REML refit', 'sigma_exact'):.0%}"` | mgcv {eval}`f"{value('n3000', 'mgcv gaulss select', 'sigma_exact'):.0%}"` |
| Error of $\log\sigma(x)$ (1,000 rows) | {eval}`round(value("n1000", "mboostrs LSS + deselect + REML refit", "log_sigma_rmse"), 3)` | mgcv {eval}`round(value("n1000", "mgcv gaulss select", "log_sigma_rmse"), 3)` |
| Error of $\mu(x)$ (1,000 rows) | {eval}`round(value("n1000", "mboostrs LSS + deselect + REML refit", "mu_rmse"), 3)` | mgcv {eval}`round(value("n1000", "mgcv gaulss select", "mu_rmse"), 3)` |

Given every column in both parameters and no hint of which matter, the
non-cyclic boosting fit followed by deselection identifies the drivers of the
mean and of the spread, and the REML refit of the kept terms estimates both
functions more accurately than mgcv's `gaulss` with term selection, in an
eighth of its time at 1,000 rows and a seventeenth at 3,000. The specification here is the generic one
(a smooth of every column in both parameters), the same for every method.

## Limitations

The noise is Gaussian and the predictors independent; a skewed or
heavy-tailed response needs another family (see
[](choosing_a_distribution.ipynb)), and correlated predictors make the
split between mean and spread harder to identify. NGBoost's trees report no
selected set here; its importance scores rank the columns but leave the cut
to the analyst. Timings are single-threaded.